In [0]:
from bs4 import BeautifulSoup
from functools import reduce
import pyspark.sql.functions as f
import pyspark.pipelines as dp
import requests
import pyspark

In [0]:

team_initials = {
    "ADB": "Admiraballs",
    "BAD": "Bad Cats",
    "BLK": "Ballard Locks",
    "CRA": "Castle Crashers",
    "DTP": "DTP",
    "DSV": "Death Savers",
    "DIH": "Drain in Hell",
    "DRK": "Drunken Dumplings",
    "ETB": "Eighteen Ball Deluxe",
    "FBP": "Flippin Big Points",
    "ICB": "Incrediballs",
    "KNR": "Knight Riders",
    "KBZ": "Kraken Ball Z",
    "RMS": "Magic Saves",
    "JMF": "Middle Flippers",
    "NMC": "Neuromancers",
    "NQG": "No Quarter Given",
    "NLT": "Northern Lights",
    "CPO": "Pants Optional",
    "PYC": "Pinballycule",
    "PGN": "Pinguins",
    "PKT": "Poketeers",
    "PBR": "Point Breakers",
    "RTR": "Ramp Tramps",
    "SSD": "Salty Sea Dogs",
    "SCN": "Seacorns",
    "SHK": "Sharks",
    "SSS": "Silver Ball Slayers",
    "SKP": "Slap Kraken Pop",
    "SWL": "Specials When Lit",
    "SRF": "Surf Champs",
    "TBT": "The B Team",
    "POW": "The Power",
    "DOG": "The Stray Dogs",
    "TTT": 'The Trailer Trashers',
    "TWC": "The Wrecking Crew",
    "TRL": "Trolls!",
    "ZOO": "Zoo Crew",
}

In [0]:
# Reusable function to scrape HTML tables and load into dataframe
def html_extract(url: str):

    html_src = requests.get(url).content
    soup = BeautifulSoup(html_src, "html.parser")
    tables = soup.find_all('table')

    # Extract data from HTML
    for table in tables:
        rows = table.find_all('tr')

        # First row contains the headers
        header_cells = rows[0].find_all(['th', 'td'])
        headers = [cell.text.strip() or f"column_{i}" for i, cell in enumerate(header_cells)]

        # Remaining rows are data
        table_data = []
        for row in rows[1:]:
            cells = row.find_all('td')
            row_dict = {headers[i]: cell.text.strip() for i, cell in enumerate(cells) if i < len(headers)}
            table_data.append(row_dict)

    return table_data

In [0]:
# Reusable function to scrape meta data for teams
def team_roster_extract(team_initial: str):

    html_src = requests.get(f"https://mondaynightpinball.com/teams/{team_initial}").content
    soup = BeautifulSoup(html_src, "html.parser")
    table = soup.find(id="team_roster")

    rows = table.find_all('tr')
    # First row contains the headers
    header_cells = rows[0].find_all(['th', 'td'])
    headers = [cell.text.strip() for cell in header_cells]

    rows = table.find_all('tr')

    # First row contains the headers
    header_cells = rows[0].find_all(['th', 'td'])
    headers = [cell.text.strip() for cell in header_cells]
    # Normalize roster column name (varies by team size: "Roster (9 Players)", "Roster (10 Players)", etc.)
    headers = ["Roster" if h.startswith("Roster") else h for h in headers]

    # Remaining rows are data
    table_data = []

    for row in rows[1:]:
        cells = row.find_all('td')
        row_dict = {headers[i]: cell.text.strip() for i, cell in enumerate(cells) if i < len(headers)}
        table_data.append(row_dict)

    return table_data

In [0]:

def create_df(team_initial: str, team_name: str):

    df = spark.createDataFrame(team_roster_extract(team_initial))
    df = df.withColumn("team_inital", f.lit(team_initial))
    df = df.withColumn("team_name", f.lit(team_name))

    return df


In [0]:
# Create an emtpy list to append dataframes to
dfs = []

for team_init, team_name in team_initials.items():
    df = create_df(team_init, team_name)
    df = df.limit(10)
    dfs.append(df)

df = reduce(pyspark.sql.dataframe.DataFrame.unionByName, dfs)

In [0]:

display(df)